In [1]:
import os
import sys
from ultralytics import RTDETR
from pathlib import Path
import torch


# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

    
from src import config

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
torch.cuda.empty_cache()

The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.
Torch: 2.8.0+cu128
CUDA available: False


In [6]:
from ultralytics import RTDETR
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

weights_path = f"{config.WORKSPACE_ROOT}/data/runs/rtdetr/train/motor_rtdetr_l_1024/weights/best.pt"

model = RTDETR(weights_path)

image_glob = "/data/horse/ws/kein254g-team_project/test_flat/**/*.jpg"
save_dir_project = "/data/horse/ws/kein254g-team_project/rtdetr_batch"
save_dir_name = "exp"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

predict_args = dict(
    source=image_glob,
    imgsz=1056,
    conf=0.3,
    device=device,
    half=False,
    batch=1,
    workers=4,
    stream=False,
    save=True,
    save_txt=True,
    save_conf=True,
    project=save_dir_project,
    name=save_dir_name,
)

try:
    results = model.predict(**predict_args)

except RuntimeError as e:
    msg = str(e)
    print("RuntimeError:", msg)
    if "CUDA out of memory" in msg or "CUDNN_STATUS_ALLOC_FAILED" in msg:
        print("\nOOM detected. Retrying with smaller settings...")
        torch.cuda.empty_cache()
        predict_args.update(dict(imgsz=768, batch=1, half=True))
        results = model.predict(**predict_args)
    else:
        raise


Torch: 2.8.0+cu128
CUDA available: False



WARNING ⚠️ imgsz=[1028] must be multiple of max stride 32, updating to [1056]
WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1100 /data/horse/ws/kein254g-team_project/test_flat/tomo_003acc_slice_0000.jpg: 1056x1056 (no detections), 4199.8ms
image 2/1100 /data/horse/ws/kein254g-team_project/test_flat/tomo_003acc_slice_0001.jpg: 1056x1056 (no detections), 4205.6ms
image 3/1100 /data/horse/ws/kein254g-team_project/test_flat/tomo_003acc_slice_0002.jpg: 1056x1056 (no detections), 4260.9ms
image 